# 三种 attention：只比较前向传播

本实验使用你的作业1.ipynb中的 PyTorch 分块和 Triton 实现，与朴素 attention 比较。它们都计算 `softmax(Q @ K.T / sqrt(d)) @ V`，不包含整个 Transformer。

统一条件：RTX 3090；B=1、d=64、FP32、非因果、无 dropout、torch.no_grad()。关闭 PyTorch TF32，并在 Triton 的前向 tl.dot 中明确指定 input_precision="ieee"。保留原实现的 tile：PyTorch 64×64，Triton 16×16。本表代表你的教学实现，不代表所有 FlashAttention 实现的性能。

计时：输入创建、正确性检查、编译和预热都在计时外。每次前向前后同步 CUDA，使用 Python default_timer 测墙钟耗时。N≤4096 预热3次、测10次；更长序列预热1次、测3次。每个形状/实现使用独立进程，退出后释放显存。

显存：单独测一次预热后的前向。总峰值为 max_memory_allocated()；额外峰值 = 总峰值 − 调用前 memory_allocated()，包含输出及临时张量。它不等于 nvidia-smi 的进程显存，也不包含 CUDA 驱动等非 PyTorch 分配。

数值检查：N≤1024 检查全部 query；更长序列检查均匀选取的64行，每行仍使用全部 K/V，容差 rtol=1e-4、atol=1e-5。OOM 和未测分别记录，不填估计值。


## 1. 找到三种实现

脚本从同目录 `作业1.ipynb` 提取定义 `FlashAttentionAutograd` 和 `FlashAttentionTriton` 的代码格，保存到结果目录的 `impl_pytorch.py`、`impl_triton.py`，不修改原 notebook。朴素版在脚本的 `def naive()` 中。请先阅读这三个函数，再观察下面的结果。

注意：原分块实现不支持任意尾块，因此本实验使用64的倍数。若要改变 d 或 block 大小，需要先检查 Triton 内核约束并重新验证正确性。


In [ ]:
from pathlib import Path
import json
import sys
import subprocess

# 在远程 cs336_systems 目录运行，使用项目 .venv 的 notebook 内核。
SCRIPT = Path("attention_forward_compare.py").resolve()
RESULT_DIR = Path("attention_forward_compare_20261026")
assert SCRIPT.exists(), "请在 cs336_systems 目录打开此 notebook"
print("Python:", sys.executable)
print("脚本:", SCRIPT)


## 2. 按需重新测试

下面代码会重新测量并覆盖相应结果；已有本次实测数据时可以跳过。保持其他 GPU 任务停止。每个组合都是独立进程，长序列的 PyTorch 双重循环可能需要数分钟。编译和预热不纳入耗时。

修改 `LENGTHS` 即可逐渐增加序列长度，其他配置保持相同。当前脚本统一固定 B=1、d=64、FP32。若要试 BF16，请作为另外一组实验修改脚本并独立保存结果，不要与此表混填。


In [ ]:
LENGTHS = [256, 1024, 2048, 4096, 8192, 16384]
RERUN = False  # 学习后改为 True 才会运行
if RERUN:
    subprocess.run([sys.executable, "-u", str(SCRIPT), "--lengths",
                    *map(str, LENGTHS)], check=True)


Linux 终端也可以执行：

```bash
cd ~/projects/diy-llm/coursework/assignment2-systems/cs336_systems
../.venv/bin/python -u attention_forward_compare.py
# 只测一个长度（仍比较三种实现）：
../.venv/bin/python -u attention_forward_compare.py --lengths 4096
```

想定位计时或显存代码：在脚本中搜索 `stage = "memory"` 和 `stage = "timing"`；搜索 `stage = "correctness"` 查看数值检查。不要在计时内部加入 profiler、打印、empty_cache 或 gc.collect。


## 3. 本次已测结果

## 前向平均耗时（ms/次）

| N | 朴素 PyTorch | PyTorch 分块 | Triton 分块 |
|---:|---:|---:|---:|
| 256 | 0.194 | 6.412 | 0.156 |
| 1024 | 0.188 | 85.772 | 0.234 |
| 2048 | 0.251 | 334.263 | 0.575 |
| 4096 | 0.666 | 1352.753 | 1.888 |
| 8192 | 2.378 | 5118.119 | 6.281 |
| 16384 | 9.840 | 22086.094 | 22.921 |

## 总峰值显存（MiB）

| N | 朴素 PyTorch | PyTorch 分块 | Triton 分块 |
|---:|---:|---:|---:|
| 256 | 8.875 | 8.488 | 8.376 |
| 1024 | 17.125 | 9.241 | 9.129 |
| 2048 | 42.125 | 10.245 | 10.133 |
| 4096 | 140.125 | 12.253 | 12.141 |
| 8192 | 528.125 | 16.269 | 16.156 |
| 16384 | 2072.125 | 24.300 | 24.188 |

## 额外峰值显存（MiB）

| N | 朴素 PyTorch | PyTorch 分块 | Triton 分块 |
|---:|---:|---:|---:|
| 256 | 0.562 | 0.176 | 0.063 |
| 1024 | 8.250 | 0.366 | 0.254 |
| 2048 | 32.500 | 0.620 | 0.508 |
| 4096 | 129.000 | 1.128 | 1.016 |
| 8192 | 514.000 | 2.144 | 2.031 |
| 16384 | 2052.000 | 4.175 | 4.062 |

## 怎么理解结果

朴素版使用 PyTorch 的矩阵乘法和 softmax，调用次数少，短序列可以很快；代价是显式保存 N×N 的分数和概率，显存随 N 近似平方增长。

PyTorch 分块版不保存整个 N×N 矩阵，因此省显存，但 Python 双重循环启动大量小算子，运行时间可能远高于朴素版。它主要展示在线 softmax 和分块算法，尚未把这些操作融合到一个 GPU 内核。

Triton 版把分块内的计算融合，避免上述大量 Python 调用，同时省去完整注意力矩阵。当前16×16配置和严格FP32计算并未针对3090调优；是否更快以表中实测为准。省显存不保证在所有形状下提速。

增大 N 时，三者仍执行近似 O(N²d) 的计算；分块减少的是中间矩阵的存储和显存读写，不是将精确 attention 的计算量变为线性。结果仅涉及推理前向，不能据此推断训练反向性能。


![耗时与额外峰值显存](attention_forward_compare_20261026/attention_comparison.png)

注意：两张图的纵轴均为对数刻度。

In [ ]:
# 读取每次采样、标准差和正确性误差，避免只看平均数。
records = [json.loads(p.read_text()) for p in RESULT_DIR.glob("*_N*.json")]
for r in sorted(records, key=lambda r: (r["N"], r["implementation"])):
    if r["status"] == "OK":
        print(f"N={r['N']:5d} {r['implementation']:16s} "
              f"{r['mean_ms']:.3f} ± {r['std_ms']:.3f} ms; "
              f"peak={r['peak_allocated_MiB']:.3f} MiB; "
              f"extra={r['extra_peak_MiB']:.3f} MiB; "
              f"max_abs_error={r['max_abs_error']:.2e}")
    else:
        print(r["N"], r["implementation"], r["status"], r.get("error", ""))
